# 03 · Baseline Models
Compares an **ESI-only** baseline, **logistic regression**, and **LightGBM**. Decision thresholds are chosen on the validation set to reach **≥90% recall** (sensitivity), prioritising missed admissions over false alarms.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
DATA = "/content/drive/MyDrive/prae/triage_subset.parquet"
df = pd.read_parquet(DATA)
df.shape

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, recall_score, precision_score)

## Features, target, split
70/15/15 stratified split (the dataset has no visit dates, so a time-based split isn't possible).

In [ ]:
y = (df["disposition"] == "Admit").astype(int)
X = df.drop(columns=["disposition"])          # never include the target or anything derived from it
cat_cols = X.select_dtypes("object").columns.tolist()
X[cat_cols] = X[cat_cols].astype("category")

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=.30, stratify=y, random_state=42)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=.50, stratify=y_tmp, random_state=42)
len(X_tr), len(X_va), len(X_te)

In [ ]:
results = []
def evaluate(name, p_va, p_te, target_recall=0.90):
    prec, rec, thr = precision_recall_curve(y_va, p_va)
    t = thr[rec[:-1] >= target_recall].max()   # threshold picked on VALIDATION
    pred = (p_te >= t).astype(int)
    r = dict(model=name, auroc=roc_auc_score(y_te, p_te), pr_auc=average_precision_score(y_te, p_te),
             threshold=t, recall=recall_score(y_te, pred), precision=precision_score(y_te, pred))
    results.append(r); return r

## Baseline 0: ESI only
The bar to beat. If the model can't beat triage acuity alone, it adds no value.

In [ ]:
esi = lambda d: d[["esi"]].fillna(3)
esi_lr = LogisticRegression().fit(esi(X_tr), y_tr)
evaluate("esi_only", esi_lr.predict_proba(esi(X_va))[:,1], esi_lr.predict_proba(esi(X_te))[:,1])

## Baseline 1: Logistic regression

In [ ]:
dum = lambda d: pd.get_dummies(d, dummy_na=True, dtype="float32")
Xd_tr = dum(X_tr)
Xd_va = dum(X_va).reindex(columns=Xd_tr.columns, fill_value=0)
Xd_te = dum(X_te).reindex(columns=Xd_tr.columns, fill_value=0)
lr = make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=1000))
lr.fit(Xd_tr, y_tr)
evaluate("logreg", lr.predict_proba(Xd_va)[:,1], lr.predict_proba(Xd_te)[:,1])

## Baseline 2: LightGBM
Handles categorical features and missing vitals natively.

In [ ]:
gbm = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63, verbose=-1)
gbm.fit(X_tr, y_tr)
evaluate("lgbm", gbm.predict_proba(X_va)[:,1], gbm.predict_proba(X_te)[:,1])

## Results

In [ ]:
pd.DataFrame(results).round(3)